# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:10])

hf_EWPVCGn


In [2]:
!pip install duckdb datasets pyarrow -q

In [3]:
import duckdb

con = duckdb.connect()

print("DuckDB is ready!")

DuckDB is ready!


In [4]:
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [5]:
REL = "hf://datasets/FlyRank/internship-warehouse"

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
feature_df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events,

    -- Engineered features
    CASE
        WHEN gsc_avg_position IS NULL THEN 1
        ELSE 0
    END AS has_missing_position,

    COALESCE(gsc_avg_position, 999) AS gsc_avg_position_filled

FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
    ga4_data_available IS TRUE
    AND gsc_data_available IS TRUE
    AND gsc_impressions > 0

LIMIT 1000
""").df()

feature_df.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events,has_missing_position,gsc_avg_position_filled
0,5,0,5.400000,1,0,0,5.400000
1,39,0,5.666667,2,0,0,5.666667
2,179,0,5.156425,2,0,0,5.156425
3,72,0,7.694444,1,0,0,7.694444
4,3282,1,6.167885,1,0,0,6.167885


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Categorical? | Available when? |
|---------|---------|------------------|--------------|-----------------|
| gsc_impressions | Number of Google Search impressions for the content on a given day. | No missing values after filtering with `gsc_data_available IS TRUE`. | No (Numeric) | Available before prediction because it is historical search performance. |
| gsc_clicks | Number of Google Search clicks for the content on a given day. | No missing values after filtering. | No (Numeric) | Available before prediction because it is measured before the prediction date. |
| gsc_avg_position | Average Google Search ranking position. Lower values indicate better ranking. | Missing values are filled using `COALESCE(..., 999)` and tracked with `has_missing_position`. | No (Numeric) | Available before prediction because it is historical ranking information. |
| ga4_sessions | Number of Google Analytics sessions for the content on a given day. | Only rows where `ga4_data_available IS TRUE` are kept. | No (Numeric) | Available before prediction because it is historical analytics data. |
| scroll_events | Number of user scroll events recorded in GA4. | Only rows with available GA4 data are used. | No (Numeric) | Available before prediction because it is collected before the decision point. |
| has_missing_position | Engineered indicator showing whether `gsc_avg_position` was missing. | Created from missing values (0 = available, 1 = missing). | Yes (Binary) | Available before prediction because it is derived from existing historical data. |
| gsc_avg_position_filled | Filled version of `gsc_avg_position` using `COALESCE`. | Missing values replaced with `999`. | No (Numeric) | Available before prediction because it is derived from historical data only. |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [10]:
potential_leakage = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "client_has_gsc",
    "client_has_ga4"
]

print("Potential leakage / non-feature columns:")

for col in potential_leakage:
    print("-", col)

safe_features = feature_df.columns.tolist()

print("\nFinal feature vector:")
print(safe_features)

Potential leakage / non-feature columns:
- client_hash_id
- content_hash_id
- report_date
- month
- client_has_gsc
- client_has_ga4

Final feature vector:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'has_missing_position', 'gsc_avg_position_filled']


### Leakage Test Result

The leakage audit identified several columns that should not be used as model features.

- `client_hash_id` and `content_hash_id` are identifiers used only for joins and grouping.
- `report_date` and `month` define the observation time and partition, not predictive signals.
- `client_has_gsc` and `client_has_ga4` describe data availability rather than content performance.

I also checked for label-derived columns (such as `trend_direction`, `trend_pct`, and `is_declining_label`). These columns are not present in the `fact_content_daily_performance` table, so no label-derived leakage was introduced.

The final feature vector contains only historical search and analytics features that are available before the prediction time.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Reason |
|----------------|--------|
| client_hash_id | Identifier only; used for joins and grouping. |
| content_hash_id | Identifier only; not a predictive feature. |
| report_date | Defines the observation date; excluded from modeling. |
| month | Dataset partition column; not a predictive feature. |
| client_has_gsc | Data availability flag; context only. |
| client_has_ga4 | Data availability flag; context only. |

## Self-check

Before you submit, confirm each line honestly:

- [ْx] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.